# Day 11 — Spark Structured Streaming Fundamentals

**Structured Streaming **is Spark's stream-processing engine. 
It allows us to process **incoming data incrementally **using familiar DataFrame and SQL operations.
Instead of loading an entire dataset each time, Spark can **process new data as it becomes available.**

### Batch V/s Streaming
````
**Batch processing	                Structured Streaming**
Processes a bounded dataset	        Processes a continuously growing dataset
Uses spark.read	                    Uses spark.readStream
Uses df.write	                    Uses df.writeStream
Executes a finite job	            Can run continuously or until available data is processed
````

Structured Streaming typically uses** micro-batches**, where Spark processes newly available data in small batches.
 when new data is available --> it create micro batches and processes the file in incremental way


Spark does not automatically launch another concurrent micro-batch for that same query after 10 seconds. The next batch starts when the current batch finishes, potentially causing processing lag.

### Trigger
Triggers control when Spark attempts to process available data.

````
   ** Trigger	                Meaning**
    availableNow=True	    Processes available data and stops
    processingTime="10      seconds"	Attempts a micro-batch approximately every 10 seconds
    Default trigger	        Processes data as quickly as possible using the engine's default scheduling behavior
````

.trigger(availableNow=True)


### What is Spark Structured Streaming, and how does it work?
Spark Structured Streaming is a stream-processing engine that allows us to process incoming data incrementally using DataFrame and SQL APIs. It typically processes data in micro-batches, where each batch handles newly available records. We use checkpoints to track processing progress and support recovery if the job fails. Depending on the use case, we can run the stream continuously or use the AvailableNow trigger for scheduled incremental processing.

**1. Which API creates a streaming DataFrame?**

A. spark.read
B. spark.readStream **--YES**
C. spark.writeStream
D. spark.streamTable

Correct - spark.readStream creates a streaming DataFrame.

**2. What does Complete output mode do for a supported streaming aggregation?**

A. Outputs only newly added records
B. Outputs only changed rows
C. Outputs the entire result table each trigger **--YES**
D. Deletes all checkpoint data

**Correct** - Complete mode emits the full current aggregation result.

**3. What happens if a micro-batch takes longer than its processing-time trigger interval?**

A. Spark automatically runs concurrent batches for the same query
B. Spark skips all incoming data
C. The next batch starts after the current batch completes **--YES**
D. The checkpoint is deleted 

**Correct**- Micro-batches within the same query execute sequentially.

# Part 2: Streaming Aggregations & Output Modes

### 2. Understanding output modes
This is one of the most frequently misunderstood Structured Streaming concepts.

**Append** --> Only newly finalized output rows
Suitable for simple streaming ingestion. For aggregations, it generally requires a watermark and a supported event-time window so Spark knows when a result is final.

**Update** --> Only rows updated since the previous trigger
For customer 1001, Spark can output a revised running total when a new transaction arrives.

**Complete** --> Entire aggregation result after each trigger
Spark outputs the current results for every customer, including customers whose totals haven't changed.


**_Visual example_**

Suppose our current aggregation is:
```
custId	totalAmount
1001	195
1002	145
1003	85
```

A new transaction of $95 arrives for customer 1001.
```
**Update mode**
custId	totalAmount
1001	290

Only changed customer
````

````
**Complete mode**
custId	totalAmount
1001	290
1002	145
1003	85

Entire updated result
````



**Important**: Output mode describes which rows a streaming query emits. It doesn't automatically specify how a destination Delta table should update existing records.

For example, if you want to maintain a customer-summary Delta table with one row per customer, you may need foreachBatch and MERGE rather than simply appending updated aggregates.

## 3. Hands-on — Streaming customer aggregations

**1. A streaming query groups transactions by custId and calculates SUM(amount), without a watermark. Which output mode is generally invalid?**
A. Complete
B. Update
C. **_Append_**

Correct- Without a way to finalize groups, Append mode cannot emit a final result for an unbounded aggregation.


**2. Which output mode emits the entire current aggregation result?**

A. Append
B. Update
C.** Complete**

Correct- Complete mode emits the full current result table after each trigger.


**3. What is the main purpose of a watermark?**

A. Automatically deduplicate all customer records
B.**_ Manage late event-time data and state cleanup for supported operations_**
C. Increase the number of streaming partitions

Correct- Watermarks help bound state for supported event-time operations and define when some late data may be discarded.

**4. A Structured Streaming aggregation groups transactions by custId and calculates SUM(amount) without a watermark.**

Which output modes can be used for the aggregation, assuming a compatible sink?

A. Append only
B.** Update and Complete**
C. Append and Update
D. All three modes

Correct!

Update and Complete are supported for this aggregation with compatible sinks. Append cannot finalize an unbounded aggregation without appropriate event-time handling.

**

Score: 3/3

**Q: What's the difference between Append, Update, and Complete output modes?** \
**Append mode** --> outputs newly finalized records, so it's commonly used for ingestion pipelines. \
**Update mode** --> outputs only the aggregation results that changed in the latest trigger. \
**Complete mode** --> outputs the entire current aggregation result every time. \
The mode we choose depends on whether we're doing simple ingestion or stateful aggregations and what the output sink supports.

**Q: Why can't we use Append mode for an aggregation without a watermark?** \
Because aggregation results can continue changing as new data arrives. Without a way to determine that a group is complete, Spark cannot safely output a final aggregate in Append mode.

Because a customer's total can continue changing whenever another transaction arrives. Spark cannot treat the aggregate as final.

# Next: Day 11 — Part 3: Watermarks and Late-Arriving Data
Before writing more code, let's understand this using a realistic payment-processing example.
1. Event time vs processing time
Suppose transactions arrive in this order:
Transaction	Event time	Arrival time	Amount
T1	10:01 AM	10:01 AM	$100
T2	10:04 AM	10:05 AM	$50
T3	10:02 AM	10:12 AM	$75
T4	10:15 AM	10:15 AM	$200

T3 is a late-arriving transaction because it occurred at 10:02 AM but wasn't received until 10:12 AM.

Streaming systems often need to aggregate transactions by when they actually happened, rather than when they arrived.
2. What is a watermark?
A watermark helps Spark determine when it can consider an event-time window sufficiently complete and remove its aggregation state.

For example:
dfStream.withWatermark("eventTime", "10 minutes")



Conceptually, Spark tracks the maximum event time it has observed and calculates a watermark approximately as:
\[
\text{Watermark} = \text{Maximum observed event time} - 10\text{ minutes}
\]

Actual watermark advancement happens through micro-batch execution.
Example: Maximum observed event time = 10:15 AM
Illustration of event-time progress. Whether a record is dropped depends on the stateful operation, its window, and when the watermark advances.

**1. What determines which 5-minute window a transaction belongs to?**

A. The time Spark processes it
B. The eventTime column
C. The checkpoint creation time
D. The Delta table modification time

Correct answer: B --> Event-time windows use the timestamp column supplied to window().

**2. What is the main purpose of a watermark?**

A. Guarantee no late data is ever dropped
B. Manage late data and bound state for supported operations
C. Deduplicate all transaction IDs
D. Force every micro-batch to execute in 10 minutes

Correct- Watermarks help Spark manage event-time state and determine when late records may be discarded.

**3. A late event arrives for a window that has not yet been finalized. What can happen?**

A. The event can update that window's aggregation
B. Spark must recreate the Delta source table
C. The event is always discarded
D. The event is automatically assigned to the current processing-time window

Correct - Eligible late events can update state for windows that remain open.


**How does Spark Structured Streaming handle late-arriving data?**
- Spark Structured Streaming supports event-time processing using watermarks. 
- We can define a watermark threshold, such as 10 minutes, to control how long Spark maintains state for supported event-time aggregations. 
- If a transaction arrives late but its window is still open, Spark can incorporate it into the aggregation. 
- Once the watermark advances beyond the window, Spark can finalize that window and discard subsequent late records.